In [96]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, SplineTransformer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Lasso
from sklearn.linear_model import LassoCV
import matplotlib.pyplot as plt

# Using Somaya's cleaning method

In [2]:
df = pd.read_csv("Loan_default.csv")
df_clean = df.copy()

In [3]:
df_clean = df_clean.rename(columns={
    "DTIRatio": "DebtToIncomeRatio"
})
# Compare the loan amount with the borrower's annual income
df_clean["LoanToIncomeRatio"] = (
    df_clean["LoanAmount"] / df_clean["Income"]
)
# LoanToIncomeRatio: Tells us how large the borrower's loan is compared to their annual income.

In [4]:
df_clean.columns

Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DebtToIncomeRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default',
       'LoanToIncomeRatio'],
      dtype='str')

# Processing Variables 

In [51]:
#Numerical 
y=df_clean['CreditScore']
num= ['Age',
            'Income',
            'MonthsEmployed',
            'NumCreditLines',
            'InterestRate',
            'LoanTerm',
            'DebtToIncomeRatio',
            'LoanToIncomeRatio']
cateogorical = ['Education',
                'EmploymentType',
                'MaritalStatus',
                'HasMortgage',
                'HasDependents',
                'LoanPurpose',
                'HasCoSigner',
                'Default']
X = df_clean[num + cateogorical]


In [52]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            num
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            cateogorical
        )
    ],
    verbose_feature_names_out=False
)
X_transformed = preprocessor.fit_transform(X)

# Do Train Test Split 

In [53]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=6021
)

# Create Models and Fit

1. Linear Model 
2. Polynomial Model Degree 2 
3. Linear Model with Lasso Regularization 
4. Polynomial Model Degree 2 with Lasso Regularization

**Did Cross Validation for Lasso to finf optimal alpha value --> used this alpha value for model 3 and 4 

Finally did cross validation for the four models AND usd test data for all four models 

In [54]:
lin_model = Pipeline([
    ("preprocessing", preprocessor),
    ("linreg", LinearRegression())
])

lin_model.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('linreg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [55]:
poly2_model = Pipeline([
    ("preprocessing", preprocessor),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("linreg", LinearRegression())
])

poly2_model.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('poly', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [ ]:
# done with claude help
poly2_lasso_cv_model = Pipeline([
    ("preprocessing", preprocessor),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LassoCV(cv=5,random_state=6021))
])

poly2_lasso_cv_model.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('poly', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [ ]:
# done with claude help

lasso = poly2_lasso_cv_model.named_steps["model"]
print("Best alpha:", lasso.alpha_)
print("Coefficients:", lasso.coef_)
print("Test R²:", poly2_lasso_cv_model.score(X_test, y_test))

Best alpha: 0.4415729905085934
Coefficients: [-4.18792724e-02 -6.64165315e-02 -1.57331990e-01  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  4.60723346e-01
  0.00000000e+00 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00
  0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
  0.00000000e+00 -1.38702377e-01 -0.00000000e+00 -0.00000000e+00
  0.00000000e+00  0.00000000e+00 -0.00000000e+00 -1.33828026e+01
 -0.00000000e+00  0.00000000e+00 -0.00000000e+00 -4.09380580e-02
 -6.90165053e-02  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00 -1.25467637e-02
 -0.00000000e+00 -0.00000000e+00  0.00000000e+00 -0.00000000e+00
  0.00000000e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00
  0.00000000e+00  2.91739642e-01 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00 -0.00000000e+00 -0.00000000e

In [73]:
lin_lasso_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Lasso(alpha=0.442))
])

lin_lasso_model.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [74]:
poly2_lasso_model = Pipeline([
    ("preprocessing", preprocessor),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", Lasso(alpha=0.442))
])

poly2_lasso_model.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('poly', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [ ]:
# Cross Validation 
models = {
    "Linear": lin_model,
    "Poly 2": poly2_model,
    "linear reg": lin_lasso_model,
    "poly2 reg": poly2_lasso_model
}

cv = KFold(n_splits=5, shuffle=True, random_state=123)

rows = []

for name, model in models.items():
    cv_rmse = -cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )

    rows.append({
        "Model": name,
        "Mean CV RMSE": cv_rmse.mean(),
        "SD CV RMSE": cv_rmse.std(ddof=1)
    })

cv_results = pd.DataFrame(rows).sort_values("Mean CV RMSE")
display(cv_results.round(2))

,Model,Mean CV RMSE,SD CV RMSE
0,Linear,158.83,0.10
2,linear reg,158.83,0.11
3,poly2 reg,158.84,0.12
1,Poly 2,158.98,0.12


In [ ]:
# Using test set 
models = {
   "Linear": lin_model,
    "Poly 2": poly2_model,
    "linear reg": lin_lasso_model,
    "poly2 reg": poly2_lasso_model
}

rows = []

for name, model in models.items():
    y_hat = model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_hat))
    test_r2 = r2_score(y_test, y_hat)

    rows.append({
        "Model": name,
        "Test RMSE": test_rmse,
        "Test R²": test_r2
    })

results = pd.DataFrame(rows)
results.round(2)

,Model,Test RMSE,Test R²
0,Linear,158.78,0.0
1,Poly 2,158.90,-0.0
2,linear reg,158.78,0.0
3,poly2 reg,158.78,0.0


# Residual plot 

In [88]:
df_clean["predicted_poly"] = poly2_model.predict(X)
df_clean["residual_poly"] = (
    df_clean["CreditScore"] - df_clean["predicted_poly"]
)

fig = px.scatter(
    df_clean,
    x="predicted_poly",
    y="residual_poly",
    title="Residual Plot: Polynomial Degree 2 Model",
    labels={
        "predicted_poly": "Predicted",
        "residual_poly": "Residual"
    }
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.show()

In [ ]:
df_clean["predicted_linear"] = lin_model.predict(X)
df_clean["residual_linear"] = (
    df_clean["CreditScore"] - df_clean["predicted_linear"]
)

fig = px.scatter(
    df_clean,
    x="predicted_linear",
    y="residual_linear",
    title="Residual Plot: Linear Model",
    labels={
        "predicted_linear": "Predicted",
        "residual_linear": "Residual"
    }
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.show()

In [90]:
df_clean["predicted_linear_lasso"] = lin_lasso_model.predict(X)
df_clean["residual_linear_lasso"] = (
    df_clean["CreditScore"] - df_clean["predicted_linear_lasso"]
)

fig = px.scatter(
    df_clean,
    x="predicted_linear_lasso",
    y="residual_linear_lasso",
    title="Residual Plot: Linear Model with Lasso",
    labels={
        "predicted_linear": "Predicted",
        "residual_linear": "Residual"
    }
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.show()

In [91]:
df_clean["predicted_poly2_lasso"] = poly2_lasso_model.predict(X)
df_clean["residual_poly2_lasso"] = (
    df_clean["CreditScore"] - df_clean["predicted_poly2_lasso"]
)

fig = px.scatter(
    df_clean,
    x="predicted_poly2_lasso",
    y="residual_poly2_lasso",
    title="Residual Plot: Polynomial Degree 2 Model with Lasso",
    labels={
        "predicted_linear": "Predicted",
        "residual_linear": "Residual"
    }
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.show()

# Figure out what variables Lasso dropped 

In [ ]:
# used claude code 

feature_names = poly2_lasso_model[:-1].get_feature_names_out()
coefs = pd.Series(lasso.coef_, index=feature_names)

dropped = coefs[coefs == 0].index.tolist()
kept = coefs[coefs != 0].index.tolist()

print(f"Dropped {len(dropped)} of {len(coefs)} features:")
print(dropped)

Dropped 303 of 324 features:
['NumCreditLines', 'InterestRate', 'LoanTerm', 'DebtToIncomeRatio', 'Education_High School', "Education_Master's", 'Education_PhD', 'EmploymentType_Part-time', 'EmploymentType_Self-employed', 'EmploymentType_Unemployed', 'MaritalStatus_Married', 'MaritalStatus_Single', 'HasMortgage_Yes', 'LoanPurpose_Business', 'LoanPurpose_Education', 'LoanPurpose_Home', 'LoanPurpose_Other', 'HasCoSigner_Yes', 'Age^2', 'Age Income', 'Age MonthsEmployed', 'Age LoanTerm', 'Age DebtToIncomeRatio', 'Age LoanToIncomeRatio', 'Age Education_High School', "Age Education_Master's", 'Age Education_PhD', 'Age EmploymentType_Part-time', 'Age EmploymentType_Self-employed', 'Age EmploymentType_Unemployed', 'Age MaritalStatus_Married', 'Age HasMortgage_Yes', 'Age HasDependents_Yes', 'Age LoanPurpose_Business', 'Age LoanPurpose_Education', 'Age LoanPurpose_Home', 'Age LoanPurpose_Other', 'Age HasCoSigner_Yes', 'Age Default_1', 'Income^2', 'Income NumCreditLines', 'Income InterestRate', 'I

In [ ]:
# used claude code 
# names coming out of the preprocessor (before polynomial expansion)
base_features = poly2_lasso_model.named_steps["preprocessing"].get_feature_names_out()

fully_dropped = [
    f for f in base_features
    if not any(f in term.split(" ") or term.startswith(f + "^") for term in kept)
]

print("Original predictors with no surviving terms:", fully_dropped)

Original predictors with no surviving terms: ['Education_High School', "Education_Master's", 'Education_PhD', 'EmploymentType_Part-time', 'EmploymentType_Self-employed', 'EmploymentType_Unemployed', 'MaritalStatus_Married', 'LoanPurpose_Business', 'LoanPurpose_Education', 'LoanPurpose_Home', 'LoanPurpose_Other', 'HasCoSigner_Yes']


In [87]:
df_kept = pd.DataFrame(kept)
print(df_kept)

                                   0
0                                Age
1                             Income
2                     MonthsEmployed
3                  LoanToIncomeRatio
4                  HasDependents_Yes
5                          Default_1
6                 Age NumCreditLines
7                   Age InterestRate
8           Age MaritalStatus_Single
9              Income MonthsEmployed
10            Income HasMortgage_Yes
11       MonthsEmployed InterestRate
12           MonthsEmployed LoanTerm
13  MonthsEmployed LoanToIncomeRatio
14  NumCreditLines DebtToIncomeRatio
15    InterestRate DebtToIncomeRatio
16    InterestRate LoanToIncomeRatio
17      InterestRate HasMortgage_Yes
18                        LoanTerm^2
19               HasDependents_Yes^2
20                       Default_1^2


# Figure out variables with right skew 

In [94]:

fig = px.scatter(
    df_clean,
    x="Income",
    y="CreditScore",
    opacity=0.65,
    title="Income Versus CreditScore",
)
fig.update_traces(marker=dict(size=2, opacity =0.2)) # claude code 
fig.show()